# m54 lab · Nuclear imaging in EP: decay, SUV and partial volume

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You work out how fluorine-18 decay enters timing and the standardized uptake value (SUV), compute SUVs, see why a
thin heart wall reads lower than its true uptake and how to correct it, measure an inflamed region on a synthetic
left ventricle, find out how often test-retest noise alone would trip a 25% response rule, and read the evidence on
PET in cardiac sarcoidosis and device infection.

| Part | What you do | Data |
| --- | --- | --- |
| A | Decay and timing (eq-m54-decay) | published half-life |
| B | The standardized uptake value (eq-m54-suv) | our own numbers |
| C | Partial volume in a wall (eq-m54-recovery) | published scanner resolution |
| D | An inflamed region on a synthetic ventricle | our own phantom |
| E | Serial scans and a 25% response rule (eq-m54-change) | published noise level and rule |
| F | Reading the evidence | published results |

Published values: the fluorine-18 half-life, 109.73 min (TUNL Nuclear Data Evaluation); the SUV definition, clinical
PET resolution of 4–7 mm FWHM, about 10% test-retest variability and the effect of clock errors (Kinahan and Fletcher
2010); the response rule of at least a 25% decrease in lesion metabolic activity (Muser et al. 2018). Doses, weights,
wall thickness, uptake values and lesion shapes are illustrative. Each part ends with **checks**; no downloads.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import nuclear as N

## Part A · Decay and timing

Activity falls as $A(t) = A_0 e^{-\lambda t}$ with $\lambda = \ln 2/T_{1/2}$ (eq-m54-decay). For fluorine-18,
$T_{1/2}$ = 109.73 min. SUV needs the injected activity corrected to the scan time, so an error $\Delta t$ in the
recorded times scales SUV by $2^{\Delta t/T_{1/2}}$.

In [ ]:
t = np.linspace(0, 360, 361)
plt.plot(t, N.activity(1.0, t))
plt.axvline(N.F18_HALF_LIFE_MIN, color="gray", ls=":")
plt.xlabel("time since injection (min)"); plt.ylabel("activity remaining (fraction)"); plt.show()
for minutes in (60, 90, 120):
    print(f"after {minutes} min: {float(N.activity(1.0, minutes)):.1%} remains")
bias10 = float(N.clock_error_bias(10.0))
print(f"a 10-min clock error biases SUV by {bias10:.1%}")

In [ ]:
# Checks
assert abs(float(N.activity(1.0, N.F18_HALF_LIFE_MIN)) - 0.5) < 1e-12
assert 0.06 < bias10 < 0.07

Kinahan and Fletcher (2010) give about 6% for a 10-minute drift, which this reproduces.

## Part B · The standardized uptake value

$\mathrm{SUV} = C/(A'/W)$: the measured concentration $C$ (kBq/mL) divided by the net injected activity $A'$,
decay-corrected to the scan time, per unit body weight $W$ (eq-m54-suv). If the tracer spread uniformly through the
body, SUV would be 1 everywhere. The dose, weight and concentrations below are illustrative.

In [ ]:
dose, weight, uptake = 370.0, 70.0, 60.0
uniform = float(N.activity(dose, uptake)) / weight
print(f"uniform distribution: SUV {float(N.suv(uniform, dose, weight, uptake)):.3f}")
c = 5.0
base = float(N.suv(c, dose, weight, uptake))
print(f"C = {c} kBq/mL: SUV {base:.2f}")
print(f"  forgetting decay correction: SUV {float(N.suv(c, dose, weight, 0.0)):.2f}")
print(f"  ignoring 2% left in the syringe: SUV {base:.2f} instead of {float(N.suv(c, dose, weight, uptake, residual_mbq=0.02 * dose)):.2f}")
print(f"  weight entered 5 kg too low: SUV {float(N.suv(c, dose, weight - 5, uptake)):.2f}")

In [ ]:
# Checks
assert abs(float(N.suv(uniform, dose, weight, uptake)) - 1.0) < 1e-12
assert float(N.suv(c, dose, weight, 0.0)) < base

## Part C · Partial volume in a wall

Clinical PET resolves about 4–7 mm FWHM (Kinahan and Fletcher 2010). Blur a slab of thickness $w$ with a Gaussian of
standard deviation $\sigma$: the maximum recovers $\operatorname{erf}(w/(2\sqrt2\sigma))$ of the true uptake and
the mean over the slab recovers $\operatorname{erf}(b) - (1 - e^{-b^2})/(b\sqrt\pi)$ with $b = w/(\sqrt2\sigma)$
(eq-m54-recovery). These are the recovery coefficients.

In [ ]:
ws = np.linspace(1, 30, 291)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for f in (4.0, 5.5, 7.0):
    s = N.fwhm_to_sigma(f)
    ax[0].plot(ws, [N.rc_peak(w, s) for w in ws], label=f"{f} mm")
    ax[1].plot(ws, [N.rc_mean(w, s) for w in ws], label=f"{f} mm")
ax[0].set_title("maximum"); ax[1].set_title("mean over the wall")
for a in ax:
    a.set_xlabel("wall thickness (mm)"); a.set_ylabel("recovery coefficient"); a.legend(title="FWHM")
plt.tight_layout(); plt.show()

s55 = N.fwhm_to_sigma(5.5)
for w in (5.0, 10.0, 15.0):
    print(f"{w:4.0f} mm wall at 5.5 mm FWHM: max {N.rc_peak(w, s55):.2f}, mean {N.rc_mean(w, s55):.2f}")
x = np.linspace(-40, 40, 80001)
prof = N.slab_profile(x, 10.0, s55)
num_mean = prof[np.abs(x) <= 5.0].mean()

In [ ]:
# Checks
assert abs(num_mean - N.rc_mean(10.0, s55)) < 1e-3
assert N.rc_mean(10.0, N.fwhm_to_sigma(7.0)) < N.rc_mean(10.0, N.fwhm_to_sigma(4.0))

The wall thicknesses here are illustrative. A wall as thick as the blur loses about a third of its mean
uptake, and one twice as thick about a sixth, even when nothing about the tissue changes.

## Part D · An inflamed region on a synthetic ventricle

A short-axis ring of myocardium 10 mm thick with uptake 2 (SUV-like units) in a background of 1, with an inflamed
arc of uptake 6 across the full wall. All values illustrative. We blur at 5.5 mm FWHM (inside the published 4–7 mm),
add 5% noise, and measure SUVmax, SUVmean and a partial-volume-corrected SUVmean (mean divided by the slab recovery
coefficient).

In [ ]:
img, xg, masks = N.lv_phantom()
rng = np.random.default_rng(54)
seen = N.add_noise(N.blur(img, xg[1] - xg[0], 5.5), 0.05, rng)
rc = N.rc_mean(10.0, s55)
true = N.lesion_stats(img, masks["lesion"])
meas = N.lesion_stats(seen, masks["lesion"], rc)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(img, origin="lower", cmap="magma", vmin=0, vmax=6.5, extent=[xg[0], xg[-1], xg[0], xg[-1]])
ax[0].set_title("true uptake")
ax[1].imshow(seen, origin="lower", cmap="magma", vmin=0, vmax=6.5, extent=[xg[0], xg[-1], xg[0], xg[-1]])
ax[1].set_title("blurred and noisy")
plt.tight_layout(); plt.show()
print(f"true: {true['mean']:.2f}")
print(f"measured: max {meas['max']:.2f}, mean {meas['mean']:.2f}, partial-volume corrected mean {meas['pvc_mean']:.2f}")

In [ ]:
# Checks
assert meas["mean"] < 0.85 * true["mean"]
assert abs(meas["pvc_mean"] - true["mean"]) < 0.15 * true["mean"]

The correction assumes we know the wall's thickness and that the lesion spans it; it also cannot remove
spill-in from neighboring tissue or the noise it amplifies. SUVmax is noisier because it rests on one voxel
(Kinahan and Fletcher 2010).

## Part E · Serial scans and a 25% response rule

In patients with cardiac sarcoidosis and VT, a decrease in lesion metabolic activity of at least 25% defined a
response on serial scans (Muser et al. 2018). Each scan carries noise; about 10% test-retest variability has been
reported for SUV (Kinahan and Fletcher 2010). With independent log-normal noise of coefficient of variation $c$ on
each scan, the measured log ratio is normal with variance $2\ln(1 + c^2)$ (eq-m54-change), so we can compute how
often noise alone flips the call.

In [ ]:
ratios = np.linspace(0.4, 1.2, 81)
for cv in (0.05, 0.10, 0.20):
    plt.plot(ratios, N.p_called_responder(ratios, cv), label=f"CV {cv:.0%}")
plt.axvline(0.75, color="gray", ls=":")
plt.xlabel("true ratio of follow-up to baseline activity"); plt.ylabel("P(called a response)"); plt.legend(); plt.show()

rng = np.random.default_rng(1)
for r in (1.0, 0.8, 0.7, 0.6):
    p = float(N.p_called_responder(r))
    sim = N.simulate_called_responder(r, 0.10, rng)
    print(f"true ratio {r:.1f}: called a response {p:.1%} (simulation {sim:.1%})")
p_false = float(N.p_called_responder(1.0))
p_miss = 1 - float(N.p_called_responder(0.7))

In [ ]:
# Checks
assert 0.015 < p_false < 0.03
assert 0.25 < p_miss < 0.4

With 10% noise per scan, an unchanged lesion is called a response about 2% of the time, but a true 30%
decrease is missed about 31% of the time: lesions near the threshold are classified by noise. The noise level is
the published test-retest figure for tumors; cardiac values may differ.

## Part F · Reading the evidence

| Study | Question | Design | Finding |
| --- | --- | --- | --- |
| Muser et al. 2016 | VT ablation in cardiac sarcoidosis: what predicts recurrence? | Single-center cohort, 31 patients | LGE, positive baseline PET and lack of PET improvement associated with recurrence |
| Muser et al. 2018 | Does change on serial FDG PET predict events? | Single-center cohort, 20 patients, 92 scans | Non-responders (LMA decrease under 25%) had more events (HR 18.96, 95% CI 2.26–159.27) |
| Bourque et al. 2024 | Nuclear imaging for cardiovascular and device infection | Multisociety expert consensus | Uptake on device pockets and leads persisting on non-attenuation-corrected images among diagnostic features |
| Birnie et al. 2014 | Arrhythmias in cardiac sarcoidosis | Expert consensus statement | Cited by title; no abstract was available to summarize |

In [ ]:
hr, lo, hi = 18.96, 2.26, 159.27
print(f"ratio of upper to lower 95% limit: {hi / lo:.0f}")
print("events-per-variable concerns: 20 patients, 12 with events (Muser et al. 2018)")

In [ ]:
# Checks
assert hi / lo > 50                       # a very wide interval: few patients, few events

A large hazard ratio with an interval spanning two orders of magnitude comes from few patients and events; it
calls for confirmation, not precise risk estimates.

## Where to go next

- In part C, at what wall thickness does the mean recover 90% at 7 mm FWHM?
- In part D, put the inflamed arc next to the blood pool and add uptake there. What does spill-in do?
- In part E, what noise level would keep the miss rate for a true 30% decrease below 10%?